# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library, referencing dataset entities by their `@id` fields for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Show high-level metadata
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Citation: {dataset.metadata.citeAs}")

## 2. Data Overview
Review available record sets and their structure using their `@id` identifiers.

In [ ]:
# List all record sets and their fields
record_sets = dataset.metadata.recordSets
if not record_sets:
    print("No record sets detected in metadata. Please check the schema definition.")
else:
    for rs in record_sets:
        print(f"Record Set @id: {rs['@id']}, name: {rs.get('name','')}\n  Fields:")
        for field in rs['fields']:
            print(f"    Field @id: {field['@id']}, name: {field.get('name','')}, dataType: {field.get('dataType','')}")

## 3. Data Extraction
Load data from a specific record set, referencing the record set and field `@id`s from the overview.

In [ ]:
# If no record sets, try to auto-detect them using dataset.records()

# Get top-level record set IDs
if not record_sets:
    print("Using fallback: auto-detecting record set IDs.")
    record_set_ids = dataset.record_set_ids
    print(f"Detected record set @ids: {record_set_ids}")
else:
    record_set_ids = [rs["@id"] for rs in record_sets]

# Preview records for each record set
dataframes = {}
for record_set_id in record_set_ids:
    print(f"---\nSample records for RecordSet @id: {record_set_id}")
    try:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print("Columns:", df.columns.tolist())
        display(df.head())
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Choose the first record set for downstream analysis
main_record_set_id = record_set_ids[0]
print(f"We select record set: {main_record_set_id} for further EDA.")
main_df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing numeric fields, and grouping. All entity references use `@id`.

In [ ]:
# Display available columns with their @id
print(f"Columns in the selected data frame: {main_df.columns.tolist()}")

# Let's auto-select a numeric field for demo; usually, you'd pick by field @id as shown in the metadata
numeric_columns = main_df.select_dtypes(include=["float", "int"]).columns.tolist()
if not numeric_columns:
    print("No numeric columns found in this main record set. Check schema and field types.")
    # Use placeholder for downstream demo
    # You may need to check the actual dataset metadata for proper field names
    numeric_field = None
else:
    numeric_field = numeric_columns[0]
    print(f"Numeric field selected for EDA: {numeric_field}")

# Filtering (if numeric)
if numeric_field:
    threshold = main_df[numeric_field].mean() if pd.notnull(main_df[numeric_field].mean()) else 0
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field}:\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to auto-detect a categorical/group field
    candidate_group_fields = [col for col in main_df.columns if col != numeric_field and main_df[col].dtype == 'object']
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(grouped_df.head())
    else:
        print("No suitable categorical field detected for grouping.")

## 5. Visualization
Visualize distributions or relationships between selected fields. All axes must use `@id` as field labels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field} (@id)')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If we found a group field earlier, show a boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=main_df)
        plt.title(f'{numeric_field} grouped by {group_field} (@id)')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We have loaded the FAIR² dataset, explored its metadata, listed the available record sets and fields by their `@id`, loaded tabular data for analysis, performed basic filtering and normalization using field `@id`s, and generated simple visualizations. For further project-specific analysis, refer to exact field names and types in the dataset's Croissant schema and use this template as a starting point.